# 🎨 Diffusion Models: Teaching AI to Paint from Noise

> **Chapter 09 — Text-to-Image Generation**  
> *Explained like you're 12, built like a staff engineer* 🧠✨

## 🧊 The Big Idea: Forward & Backward Diffusion

Imagine you have a **gorgeous ice sculpture** of a dragon. 🐉

**Forward process** = someone cranks up the heat. Over many steps, the sculpture slowly melts into a puddle of water — pure, featureless, random slush. Every step adds a tiny bit of "heat" (Gaussian noise) until there's nothing left but noise.

**Backward process** = now run the movie in reverse. Starting from a puddle of water, can we *un-melt* it back into a dragon? 🤯

That's exactly what a diffusion model learns — not to melt things down (that's easy, just add noise), but to **reverse the melting**, step by step.

### Why is this genius? 🏆

1. **We can simulate the forward process perfectly** — the math for adding Gaussian noise is trivial. This gives us infinite free training data: take any real image, noise it up, and you have a (noisy image → noise amount) training pair.

2. **The reverse is learnable** — if each forward step only adds a *tiny* amount of noise, the reverse step is also a nearly-Gaussian distribution. A neural network can learn to approximate it.

3. **Text conditioning is elegant** — we just tell the network *what* we want during the reverse steps. The network then sculpts the noise into the right shape.

| Phase | Direction | What happens | Who does it |
|-------|-----------|-------------|-------------|
| Forward | Image → Noise | Add Gaussian noise, T steps | Math (fixed) |
| Backward | Noise → Image | Remove noise, T steps | Neural network (learned) |

> 💡 **Key insight**: We never teach the model to draw from scratch. We teach it to *clean up* — one tiny step at a time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Seed for reproducibility ──────────────────────────────────────────────────
np.random.seed(42)

# ── Create a simple "image": a circle pattern on a 32x32 grid ────────────────
size = 32
x, y = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size))
r = np.sqrt(x**2 + y**2)

# A circle with a softer gradient so the diffusion is visually obvious
x0 = np.clip(1.0 - r / 0.6, 0, 1)   # bright center, dark edges
x0 = (x0 - x0.min()) / (x0.max() - x0.min())  # normalize to [0, 1]

# ── Linear noise schedule β₁ … βT ────────────────────────────────────────────
T = 10
betas = np.linspace(0.02, 0.5, T)          # β increases over time
alphas = 1.0 - betas                        # αt = 1 - βt
alpha_bar = np.cumprod(alphas)              # ᾱt = ∏_{s=1}^{t} αs

# ── Iterative forward noising ─────────────────────────────────────────────────
noisy_images = [x0]  # step 0 = clean
x_prev = x0.copy()
for t in range(T):
    noise = np.random.randn(*x0.shape)
    x_t = np.sqrt(alphas[t]) * x_prev + np.sqrt(betas[t]) * noise
    noisy_images.append(x_t)
    x_prev = x_t

# ── Plot all T+1 steps as a grid ──────────────────────────────────────────────
n_cols = T + 1
fig, axes = plt.subplots(1, n_cols, figsize=(2.0 * n_cols, 2.5))
fig.patch.set_facecolor('#1a1a2e')

step_labels = ['Clean\n(x₀)'] + [f't={i+1}' for i in range(T)]
cmap = 'plasma'

for i, (ax, img, label) in enumerate(zip(axes, noisy_images, step_labels)):
    ax.imshow(img, cmap=cmap, vmin=-1.5, vmax=1.5)
    ax.set_title(label, color='white', fontsize=8, pad=4)
    ax.axis('off')
    # Highlight start and end
    if i == 0:
        for spine in ax.spines.values():
            spine.set_edgecolor('#00ff88')
            spine.set_linewidth(2)
    if i == T:
        for spine in ax.spines.values():
            spine.set_edgecolor('#ff6b6b')
            spine.set_linewidth(2)

fig.suptitle('🧊 Forward Diffusion: Ice Sculpture → Noise Puddle',
             color='white', fontsize=13, fontweight='bold', y=1.02)

# Add an arrow annotation
fig.text(0.5, -0.05,
         '────────────────── add noise ──────────────────────►',
         ha='center', color='#aaaaaa', fontsize=10)

plt.tight_layout()
plt.savefig('/tmp/forward_diffusion.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('✅ Forward diffusion visualization complete!')
print(f'   α̅ at t=1: {alpha_bar[0]:.4f}  (most structure preserved)')
print(f'   α̅ at t={T}: {alpha_bar[-1]:.4f}  (nearly pure noise)')

## 📐 Forward Process Math — The Noise Schedule

### Noise Schedule: β₁ < β₂ < ... < βT

We add noise in **T small steps**. At each step t, we add a little Gaussian noise controlled by βt:

$$q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \mathcal{N}\!\left(\mathbf{x}_t;\; \sqrt{1-\beta_t}\,\mathbf{x}_{t-1},\; \beta_t \mathbf{I}\right)$$

Think of βt as "how much heat to apply at step t". We start small (βt ≈ 0.0001) and gradually increase (βT ≈ 0.02).

### The Reparameterization Trick 🎩

Sampling from $\mathcal{N}(\mu, \sigma^2)$ is equivalent to computing $\mu + \sigma \cdot \varepsilon$ where $\varepsilon \sim \mathcal{N}(0, I)$. This separates the randomness ($\varepsilon$) from the learned parameters — backprop flows through the mean/variance but not through the random sample.

### 🚀 The Closed-Form Shortcut: Jump Directly to Step t!

Instead of iterating T times to get xₜ, there's a beautiful closed form. Define:

$$\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$$

Then:

$$\boxed{q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}\!\left(\mathbf{x}_t;\; \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1-\bar{\alpha}_t)\mathbf{I}\right)}$$

In plain English:
$$\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\,\boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(0, I)$$

- As t → 0: $\bar{\alpha}_t ≈ 1$, so $\mathbf{x}_t ≈ \mathbf{x}_0$ (still clean)
- As t → T: $\bar{\alpha}_t ≈ 0$, so $\mathbf{x}_t ≈ \boldsymbol{\varepsilon}$ (pure noise)

This shortcut is **critical for training efficiency** — we can instantly generate a noisy version at any timestep without iterating!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# ── Setup ─────────────────────────────────────────────────────────────────────
size = 32
xg, yg = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size))
r = np.sqrt(xg**2 + yg**2)
x0 = np.clip(1.0 - r / 0.6, 0, 1)

T = 1000  # standard DDPM uses 1000 steps
betas = np.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bar = np.cumprod(alphas)   # ᾱt

def forward_closed_form(x0: np.ndarray, t: int, eps: np.ndarray) -> np.ndarray:
    """Closed-form forward process: x_t = sqrt(ᾱ_t) * x0 + sqrt(1 - ᾱ_t) * ε."""
    ab = alpha_bar[t - 1]           # ᾱ_t  (1-indexed timestep)
    return np.sqrt(ab) * x0 + np.sqrt(1.0 - ab) * eps

def forward_iterative(x0: np.ndarray, t_target: int) -> np.ndarray:
    """Iterative forward process — apply q(x_t | x_{t-1}) step by step."""
    np.random.seed(0)               # same seed so noise realizations match
    x = x0.copy()
    for t in range(1, t_target + 1):
        eps = np.random.randn(*x0.shape)
        x = np.sqrt(alphas[t - 1]) * x + np.sqrt(betas[t - 1]) * eps
    return x

# ── Test at several timesteps ─────────────────────────────────────────────────
test_steps = [1, 10, 50, 100, 250, 500, 750, 1000]

fig, axes = plt.subplots(2, len(test_steps), figsize=(2.2 * len(test_steps), 5))
fig.patch.set_facecolor('#1a1a2e')

for col, t in enumerate(test_steps):
    np.random.seed(0)
    eps = np.random.randn(*x0.shape)   # same epsilon for fair comparison

    xt_closed = forward_closed_form(x0, t, eps)
    xt_iter   = forward_iterative(x0, t)

    # Closed-form
    axes[0, col].imshow(xt_closed, cmap='plasma', vmin=-2, vmax=2)
    axes[0, col].set_title(f't={t}', color='white', fontsize=8)
    axes[0, col].axis('off')

    # Iterative
    axes[1, col].imshow(xt_iter, cmap='plasma', vmin=-2, vmax=2)
    axes[1, col].axis('off')

# Row labels
axes[0, 0].set_ylabel('Closed\nForm', color='#00ff88', fontsize=9)
axes[1, 0].set_ylabel('Iterative', color='#ff9f43', fontsize=9)

fig.suptitle('🔬 Closed-Form vs Iterative Forward Process',
             color='white', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/closed_form_vs_iterative.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

# ── Numerical verification ────────────────────────────────────────────────────
print("📊 Verification (closed-form vs iterative should look similar):")
print(f"   Note: exact pixel match won't hold because iterative noise")
print(f"   accumulates differently, but the *statistics* must match.")
print()
print("   Timestep | ᾱ_t   | Closed-form std | Iterative std")
print("   " + "-" * 52)
for t in [10, 100, 500, 1000]:
    np.random.seed(123)
    eps = np.random.randn(*x0.shape)
    xt_cf = forward_closed_form(x0, t, eps)
    xt_it = forward_iterative(x0, t)
    print(f"   t={t:<5} | {alpha_bar[t-1]:.4f} | {xt_cf.std():.4f}          | {xt_it.std():.4f}")

print()
print("✅ Both approaches produce similar noise statistics!")
print("   The closed-form is used during training — O(1) instead of O(T).")

## 🧠 Backward Process — The Neural Network Predicts Noise

### The Reverse Problem

Given $\mathbf{x}_t$ (noisy image), we want to recover $\mathbf{x}_{t-1}$ (slightly less noisy image). The true reverse distribution $q(\mathbf{x}_{t-1} | \mathbf{x}_t)$ is **intractable** — it would require knowing the entire dataset distribution. So we approximate it with a neural network:

$$p_\theta(\mathbf{x}_{t-1} \mid \mathbf{x}_t) = \mathcal{N}\!\left(\mathbf{x}_{t-1};\; \boldsymbol{\mu}_\theta(\mathbf{x}_t, t),\; \sigma_t^2 \mathbf{I}\right)$$

### Why Predict ε Instead of x₀? 🎯

Ho et al. (DDPM, 2020) showed that predicting the **noise** $\boldsymbol{\varepsilon}$ works better than predicting $\mathbf{x}_0$ directly. Here's why:

1. **Gradient magnitude**: At large t, predicting x₀ from a very noisy image involves large uncertainties. Predicting ε keeps gradients more stable.
2. **Scale invariance**: Noise is always in the same range (standard normal), regardless of what the original image looked like.
3. **Connection to score matching**: Predicting ε is equivalent to estimating the score function $\nabla_{\mathbf{x}_t} \log p(\mathbf{x}_t)$.

Once we predict $\hat{\boldsymbol{\varepsilon}}_\theta(\mathbf{x}_t, t)$, we recover the denoised mean:

$$\hat{\boldsymbol{\mu}}_\theta(\mathbf{x}_t, t) = \frac{1}{\sqrt{\alpha_t}} \left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \hat{\boldsymbol{\varepsilon}}_\theta(\mathbf{x}_t, t) \right)$$

### The Training Objective (Simplified)

$$\boxed{\mathcal{L} = \mathbb{E}_{t,\,\mathbf{x}_0,\,\boldsymbol{\varepsilon}} \left[\, \|\boldsymbol{\varepsilon} - \boldsymbol{\varepsilon}_\theta(\mathbf{x}_t, t)\|^2 \,\right]}$$

- Sample a random image $\mathbf{x}_0$ from training data
- Sample a random timestep t and random noise $\boldsymbol{\varepsilon}$
- Compute $\mathbf{x}_t$ using the closed-form formula
- Ask the network to predict $\boldsymbol{\varepsilon}$ from $(\mathbf{x}_t, t)$
- Minimize MSE between true and predicted noise

That's the entire training algorithm. Beautiful in its simplicity. 🌟

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Noise schedule (reuse from above) ────────────────────────────────────────
T = 1000
betas_t = torch.linspace(1e-4, 0.02, T)
alphas_t = 1.0 - betas_t
alpha_bar_t = torch.cumprod(alphas_t, dim=0)   # ᾱ_t, shape (T,)

def q_sample(x0: torch.Tensor, t: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    """
    Closed-form forward process:
      x_t = sqrt(ᾱ_t) * x0 + sqrt(1 - ᾱ_t) * ε

    Args:
        x0  : (B, C, H, W)  clean images
        t   : (B,)           integer timesteps in [0, T-1]
        eps : (B, C, H, W)  sampled noise
    Returns:
        x_t : (B, C, H, W)  noisy images at timestep t
    """
    ab = alpha_bar_t[t]                    # (B,)
    ab = ab.view(-1, 1, 1, 1)             # broadcast over C, H, W
    return torch.sqrt(ab) * x0 + torch.sqrt(1.0 - ab) * eps


class TinyNoisePredictor(nn.Module):
    """Toy noise predictor — a flat CNN that ignores t for simplicity.
       Real U-Nets embed t with sinusoidal position encodings."""
    def __init__(self, channels: int = 1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels + 1, 32, 3, padding=1),   # +1 for t channel
            nn.SiLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(32, channels, 3, padding=1),
        )

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Predict noise ε from noisy image x_t and timestep t.

        Args:
            x_t : (B, C, H, W)
            t   : (B,)  integer timesteps
        Returns:
            eps_pred : (B, C, H, W)
        """
        B, C, H, W = x_t.shape
        # Encode timestep as a normalized scalar channel
        t_channel = (t.float() / T).view(B, 1, 1, 1).expand(B, 1, H, W)
        x_in = torch.cat([x_t, t_channel], dim=1)   # (B, C+1, H, W)
        return self.net(x_in)


def compute_diffusion_loss(
    model: nn.Module,
    x0: torch.Tensor,
    t: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    DDPM simplified training objective:
      L = ||ε - ε_θ(x_t, t)||²

    Returns:
        loss   : scalar MSE
        eps    : true noise (B, C, H, W)
        xt     : noisy image at t (B, C, H, W)
    """
    eps = torch.randn_like(x0)          # sample true noise
    xt  = q_sample(x0, t, eps)          # compute noisy image
    eps_pred = model(xt, t)             # predict noise with network
    loss = F.mse_loss(eps_pred, eps)    # simplified ELBO objective
    return loss, eps, xt


# ── Sanity check: single forward pass ────────────────────────────────────────
torch.manual_seed(42)
B, C, H, W = 4, 1, 32, 32
model = TinyNoisePredictor(channels=C)

x0_batch = torch.rand(B, C, H, W)                         # fake "real" images
t_batch  = torch.randint(0, T, (B,))                       # random timesteps
loss, eps_true, x_noisy = compute_diffusion_loss(model, x0_batch, t_batch)

print("🧪 Single training step sanity check")
print(f"   x0 shape        : {x0_batch.shape}")
print(f"   t batch         : {t_batch.tolist()}")
print(f"   x_noisy shape   : {x_noisy.shape}")
print(f"   ε_true shape    : {eps_true.shape}")
print(f"   MSE loss        : {loss.item():.4f}  (untrained → should be ~1.0)")
print()

# ── Mini training loop: watch loss decrease ───────────────────────────────────
print("🏋️  Mini training loop (5 steps):")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
for step in range(5):
    optimizer.zero_grad()
    x0_batch = torch.rand(B, C, H, W)
    t_batch  = torch.randint(0, T, (B,))
    loss, _, _ = compute_diffusion_loss(model, x0_batch, t_batch)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    print(f"   Step {step+1}: loss = {loss.item():.4f}")

# ── Visualize: true noise vs predicted noise ──────────────────────────────────
with torch.no_grad():
    loss, eps_true, x_noisy = compute_diffusion_loss(model, x0_batch[:1], t_batch[:1])
    eps_pred = model(x_noisy, t_batch[:1])

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
fig.patch.set_facecolor('#1a1a2e')
items = [
    (x0_batch[0, 0].numpy(),    'x₀ (clean)',    'viridis'),
    (x_noisy[0, 0].numpy(),     'xₜ (noisy)',    'plasma'),
    (eps_true[0, 0].numpy(),    'ε true',         'RdBu'),
    (eps_pred[0, 0].numpy(),    'ε predicted',    'RdBu'),
]
for ax, (img, title, cmap) in zip(axes, items):
    im = ax.imshow(img, cmap=cmap)
    ax.set_title(title, color='white', fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('🎯 Training Objective: Predict the Noise',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/training_objective.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print()
print("✅ Predicted noise will look like true noise after enough training!")

## 🏗️ U-Net Architecture for Diffusion

The neural network in DDPM uses a **U-Net** — originally invented for medical image segmentation in 2015, now the backbone of Stable Diffusion.

```
Input: x_t (noisy image) + t (timestep embedding) + c (text embedding)

ENCODER (Downsampling)             DECODER (Upsampling)
──────────────────────────────────────────────────────
  [64×64×C]                              [64×64×C]  ← output ε̂
      │ ResBlock + CrossAttn          ↑
  [32×32×2C]     ──── skip ────►  [32×32×2C]
      │ ResBlock + CrossAttn          ↑  ResBlock + CrossAttn
  [16×16×4C]     ──── skip ────►  [16×16×4C]
      │ ResBlock + CrossAttn          ↑  ResBlock + CrossAttn
  [8×8×8C]       ──── skip ────►  [8×8×8C]
      │                               ↑
      └──────── Bottleneck ───────────┘
                (Self-Attn + ResBlock)
```

### Key Components

| Component | Purpose |
|-----------|--------|
| **ResBlock** | Extract spatial features, inject timestep t via AdaLayerNorm |
| **CrossAttention** | Let image features attend to text tokens (text conditioning) |
| **MaxPool / Downsample** | Reduce spatial resolution, increase channel depth |
| **Upsample** | Restore spatial resolution for output |
| **Skip Connections** | Preserve fine-grained spatial info across scales |

### Why Skip Connections? 🌉

The encoder compresses the image to capture *what* is in the scene. The decoder needs to reconstruct *where* things are. Skip connections allow the decoder to directly access the encoder's spatial maps — like cheat notes during an exam. Without them, fine details would be lost.

### Cross-Attention for Text Conditioning

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

- **Q** (queries): come from the image features — "what am I looking for?"
- **K, V** (keys/values): come from the text encoder — "here's what the text says"

Each spatial location of the image can independently ask: "which part of the text prompt is most relevant to me?" A pixel that's becoming a dog's ear will attend heavily to the word "golden retriever".

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TimestepEmbedding(nn.Module):
    """Sinusoidal timestep embedding + MLP projection."""
    def __init__(self, dim: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # Sinusoidal embedding (same as Transformers)
        half = self.mlp[0].in_features // 2
        freqs = torch.exp(
            -torch.arange(half, dtype=torch.float32, device=t.device)
            * (torch.log(torch.tensor(10000.0)) / (half - 1))
        )
        args  = t.float()[:, None] * freqs[None]
        emb   = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)
        return self.mlp(emb)


class ResBlock(nn.Module):
    """Residual block with timestep conditioning via AdaGN."""
    def __init__(self, in_ch: int, out_ch: int, t_dim: int):
        super().__init__()
        self.norm1  = nn.GroupNorm(8, in_ch)
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, out_ch * 2)  # scale + shift
        self.norm2  = nn.GroupNorm(8, out_ch)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(F.silu(self.norm1(x)))
        # Adaptive GroupNorm: modulate features with timestep
        scale, shift = self.t_proj(t_emb).chunk(2, dim=-1)
        h = h * (1 + scale[:, :, None, None]) + shift[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class CrossAttentionBlock(nn.Module):
    """Single-head cross-attention: image queries attend to text keys/values."""
    def __init__(self, img_dim: int, text_dim: int, n_heads: int = 4):
        super().__init__()
        assert img_dim % n_heads == 0, "img_dim must be divisible by n_heads"
        self.n_heads = n_heads
        self.norm    = nn.LayerNorm(img_dim)
        self.to_q    = nn.Linear(img_dim,  img_dim, bias=False)
        self.to_k    = nn.Linear(text_dim, img_dim, bias=False)
        self.to_v    = nn.Linear(text_dim, img_dim, bias=False)
        self.out_proj = nn.Linear(img_dim, img_dim)

    def forward(
        self,
        img_feat:  torch.Tensor,   # (B, C, H, W)  spatial image features
        text_emb:  torch.Tensor,   # (B, L, D_text) text token embeddings
    ) -> torch.Tensor:
        B, C, H, W = img_feat.shape
        # Flatten spatial dims → sequence
        x = img_feat.view(B, C, H * W).permute(0, 2, 1)   # (B, HW, C)
        x = self.norm(x)

        Q = self.to_q(x)          # (B, HW, C)     — from image
        K = self.to_k(text_emb)   # (B, L,  C)     — from text
        V = self.to_v(text_emb)   # (B, L,  C)     — from text

        # Multi-head split
        def split_heads(t):
            b, s, d = t.shape
            return t.view(b, s, self.n_heads, d // self.n_heads).permute(0, 2, 1, 3)

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)
        scale   = Q.shape[-1] ** -0.5
        attn    = torch.softmax(Q @ K.transpose(-2, -1) * scale, dim=-1)  # (B, h, HW, L)
        out     = (attn @ V).permute(0, 2, 1, 3).contiguous()             # (B, HW, h, d_h)
        out     = out.view(B, H * W, C)
        out     = self.out_proj(out).permute(0, 2, 1).view(B, C, H, W)

        return img_feat + out    # residual connection


class UNetBlock(nn.Module):
    """One U-Net encoder block = ResBlock + CrossAttention + MaxPool."""
    def __init__(self, in_ch: int, out_ch: int, t_dim: int, text_dim: int):
        super().__init__()
        self.res_block   = ResBlock(in_ch, out_ch, t_dim)
        self.cross_attn  = CrossAttentionBlock(out_ch, text_dim)
        self.downsample  = nn.MaxPool2d(2)

    def forward(
        self,
        x:        torch.Tensor,   # (B, in_ch, H, W)
        t_emb:    torch.Tensor,   # (B, t_dim)
        text_emb: torch.Tensor,   # (B, L, text_dim)
    ) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.res_block(x, t_emb)        # spatial features + timestep
        h = self.cross_attn(h, text_emb)    # attend to text
        skip = h                             # save before downsampling
        h    = self.downsample(h)            # halve spatial resolution
        return h, skip


# ── Smoke test ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B, IN_CH, OUT_CH, H, W = 2, 32, 64, 16, 16
T_DIM, TEXT_DIM, SEQ_LEN = 128, 768, 77

block   = UNetBlock(IN_CH, OUT_CH, T_DIM, TEXT_DIM)
t_embed = TimestepEmbedding(T_DIM)

x_in    = torch.randn(B, IN_CH, H, W)
t_in    = torch.randint(0, 1000, (B,))
text_in = torch.randn(B, SEQ_LEN, TEXT_DIM)   # e.g. CLIP text embeddings

t_emb_out       = t_embed(t_in)
h_out, skip_out = block(x_in, t_emb_out, text_in)

print("🏗️  U-Net Block — Shape Check")
print(f"   Input  x_in : {tuple(x_in.shape)}   (B, in_ch, H, W)")
print(f"   t embed     : {tuple(t_emb_out.shape)}             (B, t_dim)")
print(f"   text_in     : {tuple(text_in.shape)}   (B, seq_len, text_dim)")
print(f"   Skip (saved): {tuple(skip_out.shape)}   (B, out_ch, H, W)  ← stored for decoder")
print(f"   Output h    : {tuple(h_out.shape)}    (B, out_ch, H/2, W/2) ← downsampled")
print()
n_params = sum(p.numel() for p in block.parameters())
print(f"   Block parameters : {n_params:,}")
print()
print("✅ U-Net block working correctly!")
print("   Real models stack ~4 of these + bottleneck + symmetric decoder.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# ── Reuse CrossAttentionBlock from previous cell ──────────────────────────────

torch.manual_seed(7)

B        = 1
H, W     = 8, 8            # small spatial grid for easy visualization
IMG_DIM  = 64              # image feature channels (must be divisible by n_heads)
TEXT_DIM = 64              # text embedding dim (set equal for clarity)
N_HEADS  = 4
SEQ_LEN  = 6               # 6 text tokens: "A golden retriever puppy on grass"
TOKEN_LABELS = ["A", "golden", "retriever", "puppy", "on", "grass"]

# Simulate image features and text embeddings
img_feat_demo = torch.randn(B, IMG_DIM, H, W)
text_emb_demo = torch.randn(B, SEQ_LEN, TEXT_DIM)

# Make "golden" and "retriever" tokens have high activations
# to simulate a dog-generating scenario
text_emb_demo[0, 1] *= 3.0   # "golden"
text_emb_demo[0, 2] *= 3.0   # "retriever"

# ── Build and run cross-attention ─────────────────────────────────────────────
cross_attn_demo = CrossAttentionBlock(IMG_DIM, TEXT_DIM, n_heads=N_HEADS)

# Extract raw attention weights (before output projection)
with torch.no_grad():
    # Forward pass manually to capture attention weights
    x = img_feat_demo.view(B, IMG_DIM, H * W).permute(0, 2, 1)  # (B, HW, C)
    x_normed = cross_attn_demo.norm(x)

    Q = cross_attn_demo.to_q(x_normed)
    K = cross_attn_demo.to_k(text_emb_demo)
    V = cross_attn_demo.to_v(text_emb_demo)

    def split_h(t, n_heads):
        b, s, d = t.shape
        return t.view(b, s, n_heads, d // n_heads).permute(0, 2, 1, 3)

    Q_h = split_h(Q, N_HEADS)   # (B, H_heads, HW, d_h)
    K_h = split_h(K, N_HEADS)   # (B, H_heads, L,  d_h)
    scale = Q_h.shape[-1] ** -0.5
    attn_weights = torch.softmax(Q_h @ K_h.transpose(-2, -1) * scale, dim=-1)
    # Average across heads → (B, HW, L)
    attn_avg = attn_weights.mean(dim=1)[0]   # (HW, L)

# ── Visualize: for each text token, which image regions attend to it? ─────────
fig, axes = plt.subplots(1, SEQ_LEN, figsize=(2.5 * SEQ_LEN, 3))
fig.patch.set_facecolor('#0d1117')

for i, (ax, token) in enumerate(zip(axes, TOKEN_LABELS)):
    attn_map = attn_avg[:, i].reshape(H, W).numpy()   # which spatial locations attend to token i
    im = ax.imshow(attn_map, cmap='hot', vmin=0)
    ax.set_title(f'"{token}"', color='white', fontsize=10, pad=4)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    '🔎 Cross-Attention: Image Regions → Text Tokens\n'
    '(bright = strong attention to that word)',
    color='white', fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('/tmp/cross_attention_viz.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

print("💡 How cross-attention drives text conditioning:")
print("   • Q (queries)  ← image features: 'what should I look for?'")
print("   • K, V (keys/values) ← text tokens: 'here is what I know'")
print("   • Each pixel independently decides which words to attend to")
print("   • 'golden' and 'retriever' were amplified → see stronger maps")
print()
print(f"   Attention tensor shape: {attn_avg.shape}  (HW={H*W} spatial, L={SEQ_LEN} tokens)")
print(f"   Total cross-attn params: {sum(p.numel() for p in cross_attn_demo.parameters()):,}")

## ⚡ DiT: The Transformer Challenger to U-Net

In 2022, Peebles & Xie introduced **Diffusion Transformers (DiT)** — replacing the U-Net entirely with a Vision Transformer architecture. Stable Diffusion 3 and FLUX use DiT.

### How DiT Works

```
Input Image x_t  (B, C, H, W)
      │
  ① Patchify          → split into P×P patches → (B, N, D)  [N = HW/P²]
      │
  ② Add position embs + timestep via AdaLayerNorm
      │
  ③ L× Transformer blocks
      │  Self-Attention  (patches attend to each other)
      │  Cross-Attention (patches attend to text tokens)
      │  MLP
      │
  ④ Unpatchify         → (B, C, H, W)  ← predicted noise ε̂
```

### U-Net vs DiT: Head-to-Head 🥊

| Property | U-Net | DiT |
|----------|-------|-----|
| **Architecture** | CNN with skip connections | Pure Transformer |
| **Inductive bias** | Strong spatial (locality) | Weak (learns everything) |
| **Scalability** | Limited (hard to scale depth) | Excellent (scales like LLMs) |
| **Global context** | Only at bottleneck | Every layer, every patch |
| **Speed (same params)** | Faster (conv is efficient) | Slower but improving |
| **Quality at scale** | Great | Better (FLUX, SD3 beat SD2) |
| **Text alignment** | Cross-attn at certain layers | Cross-attn every layer |
| **Used in** | Stable Diffusion 1/2, DALL-E 2 | SD3, FLUX, Sora |

> 💡 **The trend is clear**: DiT is winning at the high end. U-Nets are still faster for smaller models. For interview prep — know both.

### Key DiT insight: Scaling Laws

DiT follows the same scaling laws as language models — double the parameters, get measurably better FID. This was *not* true for U-Nets, which is why the field is moving to DiT.

## 📝 Quiz — Chapter 09, Part 1

Test yourself! Try to answer each question before revealing the answer.

---

**Q1: What does ᾱₜ represent, and what happens to it as t → T?**

<details><summary>💡 Reveal Answer</summary>

ᾱₜ = ∏ₛ₌₁ᵗ αₛ is the cumulative product of (1 - βₛ). It controls how much of the original signal x₀ survives at timestep t. As t → T, ᾱₜ → 0, meaning the image becomes pure Gaussian noise.

</details>

---

**Q2: Why is the closed-form forward process so important for training efficiency?**

<details><summary>💡 Reveal Answer</summary>

It lets us jump directly to any timestep t in O(1) rather than running T sequential noising steps. During training we sample random t for each batch, so we'd need T forward passes per sample without the closed form — making training T× slower.

</details>

---

**Q3: Why does DDPM predict ε (noise) rather than x₀ (clean image)?**

<details><summary>💡 Reveal Answer</summary>

Predicting ε gives more stable gradients, especially at high noise levels where predicting x₀ is highly uncertain. It's also equivalent to score matching (∇ log p(xₜ)), which has strong theoretical grounding. In practice, ε-prediction produces better samples.

</details>

---

**Q4: In cross-attention, where do Q, K, and V come from?**

<details><summary>💡 Reveal Answer</summary>

Q (queries) come from the **image features** — each spatial location asks "what do I need from the text?" K (keys) and V (values) come from the **text token embeddings** — the text provides what's available. This asymmetry is what makes it *cross*-attention (vs. self-attention where Q, K, V all come from the same sequence).

</details>

---

**Q5: Name one advantage DiT has over U-Net and one advantage U-Net has over DiT.**

<details><summary>💡 Reveal Answer</summary>

**DiT advantage**: Better scaling laws — performance reliably improves with more parameters, following LLM-like scaling behavior. Also provides global context at every layer.

**U-Net advantage**: Faster inference due to convolutional efficiency and fewer FLOPs at the same parameter count. Better inductive bias for spatial tasks (locality).

</details>